# Task 1: Commentary-Based Game Analyzer
**BasketTube_RYajnik** — Deep Learning in Computer Vision

This notebook implements a Retrieval-Augmented Generation (RAG) system that answers questions **using only the game commentary**.

In [ ]:
import json
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
from ollama import chat
import time

print("✅ Libraries imported successfully")

/home/ruchikyajnik/BasketTube_RYajnik/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries imported successfully


## 1. Load the Transcript

In [2]:
import json
from pathlib import Path

# Force correct project root
project_root = Path("/home/ruchikyajnik/BasketTube_RYajnik")
transcript_path = project_root / "data/processed/transcript.json"

print(f"Looking for transcript at: {transcript_path}")

if transcript_path.exists():
    with open(transcript_path, "r", encoding="utf-8") as f:
        transcript = json.load(f)
    
    print(f"✅ Successfully loaded {len(transcript)} commentary segments")
    print(f"Total duration: ~{transcript[-1]['end']/60:.1f} minutes")
else:
    print("❌ File not found!")

Looking for transcript at: /home/ruchikyajnik/BasketTube_RYajnik/data/processed/transcript.json
✅ Successfully loaded 1930 commentary segments
Total duration: ~94.0 minutes


## 2. Build Embedding Index (Retrieval)

In [3]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [f"[{seg['start']:.2f}-{seg['end']:.2f}s] {seg['text']}" for seg in transcript]
embeddings = embedder.encode(texts, normalize_embeddings=True)

print("✅ Embedding index created for fast retrieval")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1513.31it/s]


✅ Embedding index created for fast retrieval


## 3. QA Function

In [4]:
def ask_commentary(question, top_k=6):
    start_time = time.time()
    
    # Retrieve most relevant segments
    query_emb = embedder.encode(question, normalize_embeddings=True)
    scores = np.dot(embeddings, query_emb)
    top_idx = scores.argsort()[-top_k:][::-1]
    
    context = "\n".join([texts[i] for i in top_idx])
    
    system_prompt = """You are a professional basketball analyst.
Answer ONLY using the provided commentary.
Always cite timestamps (e.g., [123.45-125.67s]).
Structure your answer clearly.
Separate direct facts from interpretations.
Be concise and professional."""

    response = chat(
        model="qwen2.5:14b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Commentary:\n{context}\n\nQuestion: {question}"}
        ]
    )
    
    answer = response['message']['content']
    elapsed = time.time() - start_time
    
    print(f"\n💬 Question: {question}")
    print("=" * 90)
    print(answer)
    print("=" * 90)
    print(f"⏱️  Answered in {elapsed:.1f} seconds")
    print("Note: This is based solely on commentary and may not reflect actual game events.")
    
    return answer

## 4. Example Questions & Answers

In [5]:
questions = [
    "How did LeBron perform?",
    "What happened when LeBron got poked in the eye?",
    "What was said about shooting or performance in the second half?",
    "Summarize the key comments about team performance.",
    "Any clutch moments or big plays mentioned?"
]

for q in questions:
    ask_commentary(q)
    print("\n")


💬 Question: How did LeBron perform?
**Facts:**
- LeBron James was visibly affected by a shot from Draymond Green, indicating he experienced physical discomfort during the game ([5411.69-5419.38s]).
- He sat out for some part of the game ([4326.40-4328.04s]).

**Interpretations:**
- LeBron James had a significant impact on the game, being described as "absolutely sensational" and drawing substantial attention to his performance ([1160.35-1166.87s]).
- His return to the court after sitting out was noted more than once, emphasizing his importance to the team's dynamics ([2154.62-2156.40s], [3636.70-3638.22s]).
⏱️  Answered in 11.3 seconds
Note: This is based solely on commentary and may not reflect actual game events.



💬 Question: What happened when LeBron got poked in the eye?
Direct facts:
- Draymond Green poked LeBron James in the eye during a game ([123.45-125.67s], [5334.94-5344.27s]).
- LeBron had difficulty seeing after being poked ([123.45-125.67s], [5334.94-5344.27s]).

Interp

## Methodology

This implementation of **Task 1: Analyzing Player Performance from Commentary** was designed as a robust, fully local Retrieval-Augmented Generation (RAG) system.

### Core Architecture
1. **Data Ingestion**
   - Loaded 1,930 timestamped commentary segments from `data/processed/transcript.json` (generated via Faster-Whisper `large-v3` model).
   - Each segment contains precise start/end timestamps and raw text.

2. **Embedding & Retrieval**
   - Used `SentenceTransformer("all-MiniLM-L6-v2")` to generate 384-dimensional embeddings for all segments.
   - Retrieval performed via cosine similarity with `top_k=6` most relevant segments.
   - This lightweight embedding model was chosen for its excellent balance of speed and semantic quality on a local machine.

3. **Generation Layer**
   - Local LLM: `qwen2.5:14b` via Ollama.
   - Strict system prompt engineered to enforce grounding, timestamp citation, and professional tone.
   - No external APIs used — everything runs locally for reproducibility and zero cost.

### Justification for This Approach
While the official starter Colab focuses heavily on advanced vision models (RF-DETR, SAM2, SmolVLM2), Task 1 is fundamentally a **text-based RAG problem**. 

My local RAG approach was deliberately chosen because:
- It is computationally efficient and runs smoothly on consumer hardware (RTX 5070 Ti).
- It achieves strong grounding and citation quality without the complexity and installation headaches of vision-heavy pipelines.
- It directly fulfills the assignment’s core requirement: answering questions **using only the spoken commentary**.
- It demonstrates core Deep Learning concepts (dense retrieval, prompt engineering, grounded generation) cleanly and effectively.

This pragmatic approach prioritizes reliability, interpretability, and educational value over chasing the most complex vision stack for a text-based task.

## Limitations

Despite strong overall performance, several limitations exist:

1. **Retrieval Quality**
   - Relies on semantic similarity; very specific or rare terms may not always retrieve the best segments.
   - Long transcripts can dilute precision in some edge cases.

2. **LLM Grounding**
   - Even with a strict prompt, the 14B model occasionally produces minor hallucinations or overly interpretive answers.
   - Timestamp citation is generally good but not perfect.

3. **Lack of Visual Verification**
   - Purely commentary-based — cannot confirm whether commentators were accurate (this is addressed in Task 2).

4. **No Multi-Turn Memory**
   - Current implementation is stateless per question (no conversation history).

5. **Model Size Trade-off**
   - Chose 14B model for quality; smaller models (7B) were faster but noticeably less coherent.

These limitations are acceptable given the local-only constraint and the strong educational focus of the project.